# Inspect a saved rigid-motif decomposition

This notebook loads completed QM9, GEOM-Drugs, or QMugs preprocessing artifacts, maps a retained fragmentation record back to its aligned molecule, and displays the saved rigid-motif assignment. It never mines or modifies a vocabulary.

In [ ]:
from pathlib import Path

from IPython.display import display

from motiflow.visualization import (
    fragmentation_figure,
    fragmentation_summary,
    fragmentation_table,
    load_fragmentation_example,
)

## Select an artifact

Choose `DATASET` from `qm9`, `geom`, and `qmugs`. The fragmented directory is derived from the standard output layout. `PROCESSED_INDEX` addresses a retained molecule in the selected fragmented split; the summary reports its original aligned index and source sample ID. Set `SHOW_ATOM_LABELS` to `None` for automatic labels on small molecules only.

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "configs").is_dir():
    raise RuntimeError("Start Jupyter from the MotiFlow repository or its notebooks directory.")

DATASET = "geom"
SPLIT = "train"
PROCESSED_INDEX = 1000
ALPHA_THRESHOLD = 0.1
SHOW_ATOM_LABELS = None

alpha_slug = f"{ALPHA_THRESHOLD:g}".replace(".", "p")
ALIGNED_ROOT = PROJECT_ROOT / "data/preprocessed"
FRAGMENTED_DIR = PROJECT_ROOT / "data/fragmented" / DATASET / f"alpha_{alpha_slug}"

if DATASET not in {"qm9", "geom", "qmugs"}:
    raise ValueError(f"Unknown dataset {DATASET!r}.")
if not (FRAGMENTED_DIR / "fragment_library.pt").is_file():
    available = sorted(
        str(path.relative_to(PROJECT_ROOT / "data/fragmented"))
        for path in (PROJECT_ROOT / "data/fragmented").glob("*/*")
        if (path / "fragment_library.pt").is_file()
    )
    raise FileNotFoundError(
        f"No completed artifacts at {FRAGMENTED_DIR}. Available artifact directories: {available}"
    )

print(f"Dataset: {DATASET}; split: {SPLIT}; artifacts: {FRAGMENTED_DIR}")

In [ ]:
example = load_fragmentation_example(
    aligned_root=ALIGNED_ROOT,
    fragmented_dir=FRAGMENTED_DIR,
    dataset=DATASET,
    split=SPLIT,
    processed_index=PROCESSED_INDEX,
)

for key, value in fragmentation_summary(example).items():
    print(f"{key:>24}: {value}")

display(fragmentation_table(example))

In [ ]:
fragmentation_figure(
    example,
    show_atom_labels=SHOW_ATOM_LABELS,
).show()

For dense GEOM-Drugs and QMugs molecules, hover over atoms for their element, atom index, fragment, and vocabulary class. Legend entries can be clicked to isolate or hide individual rigid motifs; dotted gray lines are bonds cut between motifs.